In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\Users\anuro\online_retail_II.csv",
    encoding="ISO-8859-1"
)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

missing

In [ ]:
df.duplicated().sum()

In [ ]:
(df["Quantity"] < 0).sum()

In [ ]:
(df["Quantity"] == 0).sum()

In [ ]:
(df["Price"] <= 0).sum()

In [ ]:
df[df["Quantity"] < 0].head(10)

In [ ]:
df[df["Price"] <= 0].head(10)

In [ ]:
df[df["Invoice"].str.startswith("C", na=False)].head(10)

In [ ]:
# Create a working copy of the original dataset
retail = df.copy()

# Convert InvoiceDate into datetime format
retail["InvoiceDate"] = pd.to_datetime(
    retail["InvoiceDate"],
    errors="coerce"
)

# Create a revenue column
retail["Revenue"] = retail["Quantity"] * retail["Price"]

# Display the first five rows
retail.head()

In [ ]:
retail[[
    "Invoice",
    "Description",
    "Quantity",
    "Price",
    "Revenue"
]].head(10)

In [ ]:
total_revenue = retail["Revenue"].sum()

print("Total Revenue:", round(total_revenue, 2))

In [ ]:
total_quantity = retail["Quantity"].sum()

print("Total Net Quantity:", total_quantity)

In [ ]:
total_invoices = retail["Invoice"].nunique()

print("Total Unique Invoices:", total_invoices)

In [ ]:
# Extract year and month
retail["YearMonth"] = retail["InvoiceDate"].dt.to_period("M")

# Calculate monthly revenue
monthly_revenue = (
    retail.groupby("YearMonth")["Revenue"]
    .sum()
    .reset_index()
)

# Display the first 12 months
monthly_revenue.head(12)

In [ ]:
highest_month = monthly_revenue.loc[
    monthly_revenue["Revenue"].idxmax()
]

print("Highest Revenue Month:", highest_month["YearMonth"])
print("Revenue:", round(highest_month["Revenue"], 2))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(
    monthly_revenue["YearMonth"].astype(str),
    monthly_revenue["Revenue"],
    marker="o"
)

plt.title("Monthly Revenue Analysis")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=90)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(
    monthly_revenue["YearMonth"].astype(str),
    monthly_revenue["Revenue"],
    marker="o"
)

plt.title("Monthly Revenue Trend", fontsize=16)
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
product_revenue = (
    retail.groupby("Description")["Revenue"]
    .sum()
    .reset_index()
)

product_revenue = product_revenue.sort_values(
    by="Revenue",
    ascending=False
)

product_revenue.head(10)

In [ ]:
top10_products = product_revenue.head(10)

plt.figure(figsize=(12, 6))

plt.barh(
    top10_products["Description"],
    top10_products["Revenue"]
)

plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Product")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
country_revenue = (
    retail.groupby("Country")["Revenue"]
    .sum()
    .reset_index()
)

country_revenue = country_revenue.sort_values(
    by="Revenue",
    ascending=False
)

country_revenue.head(10)

In [ ]:
import matplotlib.pyplot as plt

top10_countries = country_revenue.head(10)

plt.figure(figsize=(12, 6))

plt.barh(
    top10_countries["Country"],
    top10_countries["Revenue"]
)

plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Country")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
customer_revenue = (
    retail.dropna(subset=["Customer ID"])
    .groupby("Customer ID")["Revenue"]
    .sum()
    .reset_index()
)

customer_revenue = customer_revenue.sort_values(
    by="Revenue",
    ascending=False
)

customer_revenue.head(10)

In [ ]:
import matplotlib.pyplot as plt

top10_customers = customer_revenue.head(10)

plt.figure(figsize=(12, 6))

plt.barh(
    top10_customers["Customer ID"].astype(str),
    top10_customers["Revenue"]
)

plt.title("Top 10 Customers by Revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("Customer ID")

plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
unique_customers = retail["Customer ID"].nunique()

print("Total Unique Customers:", unique_customers)

In [ ]:
average_order_value = (
    retail.groupby("Invoice")["Revenue"].sum().mean()
)

print("Preliminary Average Invoice Value:",
      round(average_order_value, 2))

In [ ]:
# Create a separate working copy
retail_clean = retail.copy()

# Record the original row count
rows_before = len(retail_clean)

# Remove exact duplicate rows
retail_clean = retail_clean.drop_duplicates().copy()

# Record the new row count
rows_after = len(retail_clean)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Duplicates removed:", rows_before - rows_after)

In [ ]:
print("Remaining duplicates:", retail_clean.duplicated().sum())
print("Dataset shape:", retail_clean.shape)

In [ ]:
# Identify cancellation invoices
cancellations = retail_clean[
    retail_clean["Invoice"].str.startswith("C", na=False)
]

print("Cancellation records:", len(cancellations))
print("Unique cancellation invoices:", cancellations["Invoice"].nunique())

In [ ]:
cancellations[[
    "Invoice",
    "Description",
    "Quantity",
    "Price",
    "Revenue"
]].head(10)

In [ ]:
zero_price = retail_clean[retail_clean["Price"] == 0]

negative_price = retail_clean[retail_clean["Price"] < 0]

print("Zero-price records:", len(zero_price))
print("Negative-price records:", len(negative_price))

In [ ]:
zero_price[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "Customer ID",
        "Country"
    ]
].head(15)

In [ ]:
negative_price[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "Customer ID",
        "Country"
    ]
]

In [ ]:
zero_price[["Quantity", "Price"]].describe()

In [ ]:
zero_price["Description"].value_counts(dropna=False).head(15)

In [ ]:
negative_price[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "Customer ID",
        "Country"
    ]
]

In [ ]:
sales_analysis = retail_clean[
    (retail_clean["Quantity"] > 0) &
    (retail_clean["Price"] > 0)
].copy()

print("Original cleaned rows:", len(retail_clean))
print("Positive-sales rows:", len(sales_analysis))
print("Rows excluded from this analysis:", 
      len(retail_clean) - len(sales_analysis))

In [ ]:
sales_analysis[[
    "Invoice",
    "Description",
    "Quantity",
    "Price",
    "Revenue"
]].head(10)

In [ ]:
positive_sales_revenue = sales_analysis["Revenue"].sum()

print(
    "Positive-Transaction Revenue:",
    round(positive_sales_revenue, 2)
)

In [ ]:
# Check missing values in the sales-analysis dataset
print("Missing values:")
print(sales_analysis.isnull().sum())

# Check duplicate records
print("\nRemaining duplicates:", sales_analysis.duplicated().sum())

# Check date range
print("\nFirst transaction:", sales_analysis["InvoiceDate"].min())
print("Last transaction:", sales_analysis["InvoiceDate"].max())

# Check revenue calculation
print("\nRevenue validation:")
print(
    (sales_analysis["Quantity"] * sales_analysis["Price"])
    .equals(sales_analysis["Revenue"])
)

In [ ]:
# Save the prepared sales-analysis dataset
sales_analysis.to_csv(
    r"C:\Users\anuro\ecommerce_sales_clean.csv",
    index=False
)

print("Dataset saved successfully!")
print("Rows:", len(sales_analysis))
print("Columns:", len(sales_analysis.columns))

In [ ]:
# Create a separate copy for MySQL import
mysql_data = sales_analysis.copy()

# Ensure missing Customer IDs remain missing
mysql_data["Customer ID"] = mysql_data["Customer ID"].astype("Int64")

# Save the new CSV
mysql_data.to_csv(
    r"C:\Users\anuro\ecommerce_sales_mysql.csv",
    index=False,
    na_rep=""
)

print("MySQL import file created!")
print("Rows:", len(mysql_data))

In [ ]:
import mysql.connector

print("MySQL connector imported successfully!")

In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=input("Enter your MySQL password: "),
    database="ecommerce_sales_db"
)

print("Connected to MySQL successfully!")

In [ ]:
import pandas as pd
import mysql.connector
from mysql.connector import Error

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=input("Enter your MySQL password: "),
    database="ecommerce_sales_db"
)

cursor = conn.cursor()

# Use the prepared DataFrame
data = sales_analysis.copy()

# Convert missing values to None for MySQL
data = data.astype(object).where(pd.notna(data), None)

columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country",
    "Revenue", "YearMonth"
]

insert_sql = """
INSERT INTO ecommerce_sales_staging
(Invoice, StockCode, Description, Quantity, InvoiceDate,
 Price, `Customer ID`, Country, Revenue, YearMonth)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

# Prevent accidental duplicate imports
cursor.execute("TRUNCATE TABLE ecommerce_sales_staging")

batch_size = 5000
total_rows = len(data)

for start in range(0, total_rows, batch_size):
    batch = data.iloc[start:start + batch_size][columns]
    records = list(batch.itertuples(index=False, name=None))

    cursor.executemany(insert_sql, records)
    conn.commit()

    completed = min(start + batch_size, total_rows)
    print(f"Imported {completed:,} / {total_rows:,} rows")

print("Import completed successfully!")

cursor.close()
conn.close()

In [ ]:
import pandas as pd

retail = pd.read_csv(
    r"C:\Users\anuro\ecommerce_sales_clean.csv"
)

print("Dataset loaded successfully!")
print("Rows:", len(retail))
print("Columns:", len(retail.columns))

retail.head()

In [ ]:
sales_analysis = retail.copy()

# Confirm the dataset is available
print("Sales analysis dataset created!")
print("Rows:", len(sales_analysis))
print("Columns:", len(sales_analysis.columns))

In [ ]:
print(sales_analysis.columns.tolist())

In [ ]:
print("Total rows:", len(sales_analysis))
print("Total columns:", len(sales_analysis.columns))
print("Missing Customer IDs:", sales_analysis["Customer ID"].isna().sum())

In [ ]:
import mysql.connector
from getpass import getpass

password = getpass("Enter your MySQL password: ")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="ecommerce_sales_db"
)

print("Connected to MySQL successfully!")

In [ ]:
import pandas as pd
import mysql.connector

# Prepare the data
data = sales_analysis.copy()

# Convert missing values to None for MySQL
data = data.astype(object).where(pd.notna(data), None)

columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country",
    "Revenue", "YearMonth"
]

insert_sql = """
INSERT INTO ecommerce_sales_staging
(
    Invoice, StockCode, Description, Quantity, InvoiceDate,
    Price, `Customer ID`, Country, Revenue, YearMonth
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor = conn.cursor()

# Clear the incomplete import
cursor.execute("TRUNCATE TABLE ecommerce_sales_staging")
conn.commit()

# Import in batches
batch_size = 5000
total_rows = len(data)

for start in range(0, total_rows, batch_size):
    batch = data.iloc[start:start + batch_size][columns]
    records = list(batch.itertuples(index=False, name=None))

    cursor.executemany(insert_sql, records)
    conn.commit()

    completed = min(start + batch_size, total_rows)
    print(f"Imported {completed:,} / {total_rows:,} rows")

print("Import completed successfully!")

cursor.close()
conn.close()